# Positional Encoding

Kod, kavramları net bir şekilde ayırmak için bölümlere ayrılmıştır ve her adımda ne yapıldığını açıklayan yorum satırları içerir.

**Kodun Yapısı ve Anlatımı**

* Bölüm 1, 2 ve 3 (Sinusoidal, Learned, RoPE): Bu yöntemler, kelime vektörlerini veya Query/Key vektörlerini attention mekanizmasına girmeden önce modifiye eder. Kod, basit bir "elma" kelimesi için örnek bir embedding vektörü oluşturur ve bu üç yöntemin, kelime farklı pozisyonlardayken bu vektörü nasıl değiştirdiğini gösterir.

* Bölüm 4 (Relative Position Bias - T5 Stili): Bu yöntem diğerlerinden temel olarak farklıdır. Girdi vektörlerini değiştirmez. Bunun yerine, attention mekanizmasının içinde, Query ve Key vektörleri arasındaki attention skorları hesaplandıktan sonra, bu skorlara doğrudan bir "konum yanlılığı" (position bias) ekler. Bu nedenle bu bölümdeki kod, diğerlerinden farklı olarak bir attention skoru matrisini simüle ederek çalışır.

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# --- Genel Ayarlar ve Hazırlık ---
print("### 4 Farklı Konumsal Kodlama (Positional Encoding) Yönteminin Karşılaştırılması ###")
D_MODEL = 64      # Modelin embedding boyutu
MAX_SEQ_LEN = 100 # Maksimum dizi uzunluğu
sns.set_theme(style="whitegrid")

# Tüm örneklerde kullanılacak, "elma" kelimesini temsil eden basit bir vektör.
# Amacımız, PE sonrası değişimi net görmektir.
example_word_embedding = torch.ones(D_MODEL)
pos_1 = 5   # "elma" kelimesinin 5. pozisyonda olduğunu varsayalım
pos_2 = 12  # "elma" kelimesinin 12. pozisyonda olduğunu varsayalım

print(f"\nModel Embedding Boyutu (d_model): {D_MODEL}")
print(f"Karşılaştırılacak Pozisyonlar: {pos_1} ve {pos_2}")
print("-" * 60)

# ==============================================================================
# BÖLÜM 1: Sinusoidal Positional Encoding (Orijinal)
# Tip: Sabit, Konum: Mutlak, Uygulama: Embedding'e eklenir
# ==============================================================================
print("\n### BÖLÜM 1: Sinusoidal Positional Encoding ###")
print("Özellikler: Sabit, Mutlak Konum, İyi Genelleme Yeteneği")

def get_sinusoidal_pe(max_len, d_model):
    pe = torch.zeros(max_len, d_model)
    position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
    div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-np.log(10000.0) / d_model))
    pe[:, 0::2] = torch.sin(position * div_term)
    pe[:, 1::2] = torch.cos(position * div_term)
    return pe

sinusoidal_pe_matrix = get_sinusoidal_pe(MAX_SEQ_LEN, D_MODEL)
pe_pos_1 = sinusoidal_pe_matrix[pos_1]
pe_pos_2 = sinusoidal_pe_matrix[pos_2]

# Adım adım çıktı
print(f"\nAdım 1.1: 'elma' kelimesinin başlangıç vektörü (özet):\n{example_word_embedding[:5]}...")
print(f"\nAdım 1.2: Pozisyon {pos_1} için Sinusoidal PE vektörü (özet):\n{pe_pos_1[:5]}...")
print(f"\nAdım 1.3: Pozisyon {pos_2} için Sinusoidal PE vektörü (özet):\n{pe_pos_2[:5]}...")

final_embedding_1 = example_word_embedding + pe_pos_1
final_embedding_2 = example_word_embedding + pe_pos_2

print(f"\nSONUÇ 1: Pozisyon {pos_1} için son vektör (özet):\n{final_embedding_1[:5]}...")
print(f"SONUÇ 2: Pozisyon {pos_2} için son vektör (özet):\n{final_embedding_2[:5]}...")
print("Görüldüğü gibi, aynı kelime vektörü farklı pozisyonlar için farklı son vektörlere dönüştü.")
print("-" * 60)


# ==============================================================================
# BÖLÜM 2: Learned Absolute Positional Encoding
# Tip: Öğrenilmiş, Konum: Mutlak, Uygulama: Embedding'e eklenir
# ==============================================================================
print("\n### BÖLÜM 2: Learned Absolute Positional Encoding ###")
print("Özellikler: Öğrenilmiş, Mutlak Konum, Zayıf Genelleme Yeteneği")

# Her pozisyon için öğrenilebilir bir vektör tutan tablo (layer)
learned_pe_layer = nn.Embedding(MAX_SEQ_LEN, D_MODEL)

# Pozisyonları tensor olarak hazırlayalım
pos_tensor_1 = torch.tensor([pos_1], dtype=torch.long)
pos_tensor_2 = torch.tensor([pos_2], dtype=torch.long)

# Adım adım çıktı
learned_pe_pos_1 = learned_pe_layer(pos_tensor_1).squeeze(0)
learned_pe_pos_2 = learned_pe_layer(pos_tensor_2).squeeze(0)

print(f"\nAdım 2.1: Pozisyon {pos_1} için (rastgele) öğrenilmiş PE vektörü (özet):\n{learned_pe_pos_1.data[:5]}...")
print(f"\nAdım 2.2: Pozisyon {pos_2} için (rastgele) öğrenilmiş PE vektörü (özet):\n{learned_pe_pos_2.data[:5]}...")

final_learned_embedding_1 = example_word_embedding + learned_pe_pos_1
final_learned_embedding_2 = example_word_embedding + learned_pe_pos_2

print(f"\nSONUÇ 1: Pozisyon {pos_1} için son vektör (özet):\n{final_learned_embedding_1.data[:5]}...")
print(f"SONUÇ 2: Pozisyon {pos_2} için son vektör (özet):\n{final_learned_embedding_2.data[:5]}...")
print("Bu PE vektörleri, eğitim sırasında veriye göre optimize edilir.")
print(f"Not: Bu yöntem, MAX_SEQ_LEN ({MAX_SEQ_LEN}) dışındaki pozisyonları bilemez.")
print("-" * 60)

# ==============================================================================
# BÖLÜM 3: Rotary Positional Encoding (RoPE)
# Tip: Sabit, Konum: Göreceli (Örtük), Uygulama: Q/K vektörleri döndürülür
# ==============================================================================
print("\n### BÖLÜM 3: Rotary Positional Encoding (RoPE) ###")
print("Özellikler: Sabit, Göreceli Konum, Mükemmel Genelleme Yeteneği")
print("Not: RoPE, vektör eklemek yerine Query ve Key vektörlerini 'döndürür'.")

def get_rope_angles(positions, d_model):
    angle_rads = 1 / (10000 ** (torch.arange(0, d_model, 2).float() / d_model))
    return positions.unsqueeze(1) * angle_rads.unsqueeze(0)

def apply_rotary_pos_emb(x, angles):
    cos_val = torch.cos(angles).repeat_interleave(2, dim=-1)
    sin_val = torch.sin(angles).repeat_interleave(2, dim=-1)
    x_rotated_part = torch.cat([-x[..., 1::2], x[..., 0::2]], dim=-1)
    return x * cos_val + x_rotated_part * sin_val

# Adım adım çıktı
dummy_query_vector = torch.randn(D_MODEL) # Örnek bir Query vektörü
print(f"\nAdım 3.1: Örnek bir Query vektörü (özet):\n{dummy_query_vector[:5]}...")

angles_pos_1 = get_rope_angles(torch.tensor([pos_1], dtype=torch.float), D_MODEL)
rotated_query_1 = apply_rotary_pos_emb(dummy_query_vector, angles_pos_1)

angles_pos_2 = get_rope_angles(torch.tensor([pos_2], dtype=torch.float), D_MODEL)
rotated_query_2 = apply_rotary_pos_emb(dummy_query_vector, angles_pos_2)

print(f"\nSONUÇ 1: Pozisyon {pos_1} için döndürülmüş Query (özet):\n{rotated_query_1.squeeze(0)[:5]}...")
print(f"SONUÇ 2: Pozisyon {pos_2} için döndürülmüş Query (özet):\n{rotated_query_2.squeeze(0)[:5]}...")
print("RoPE'un asıl gücü, iki vektör arasındaki attention skorunun göreceli mesafeye bağlı olmasıdır.")
print("-" * 60)

# ==============================================================================
# BÖLÜM 4: Relative Position Bias (T5 Stili)
# Tip: Öğrenilmiş, Konum: Göreceli (Açık), Uygulama: Attention skorlarına eklenir
# ==============================================================================
print("\n### BÖLÜM 4: Relative Position Bias (T5 Stili) ###")
print("ÖZELLİKLE DİKKAT: Bu yöntem diğerlerinden farklı çalışır.")
print("Vektörleri DEĞİŞTİRMEZ, bunun yerine Attention skor matrisine öğrenilmiş bir 'yanlılık' (bias) ekler.")

# T5'in kullandığı standart parametreler
num_buckets = 32
max_distance = 128
# Bu bias değerleri her bir attention başlığı için ayrı ayrı öğrenilir, biz 1 başlık varsayalım.
relative_attention_bias_table = nn.Embedding(num_buckets, 1)

def compute_t5_relative_bias(seq_len):
    # Adım 4.1: Query ve Key pozisyonları arasındaki göreceli mesafeyi hesapla
    # Sonuç: `context_position[i, j] = i - j`
    context_position = torch.arange(seq_len, dtype=torch.long)[:, None]
    memory_position = torch.arange(seq_len, dtype=torch.long)[None, :]
    relative_position = memory_position - context_position

    # Adım 4.2: Bu mesafeleri "bucket" (kova) denen gruplara ayır.
    # T5'in orjinalindeki karmaşık logaritmik kovalama yerine, anlaşılır bir versiyon kullanalım.
    rp_bucket = torch.zeros_like(relative_position)
    
    # Pozitif ve negatif mesafeler için farklı kovalama
    num_buckets_half = num_buckets // 2
    rp_bucket += (relative_position > 0).long() * num_buckets_half
    
    # Mutlak mesafeyi al ve sınırla
    relative_position_abs = torch.abs(relative_position)
    max_exact = num_buckets_half // 2
    
    # Yakın mesafeleri doğrudan kovaya ata
    is_small = relative_position_abs < max_exact
    rp_bucket[is_small] += relative_position_abs[is_small]
    
    # Uzak mesafeleri logaritmik olarak grupla
    is_large = ~is_small
    val_large = max_exact + (torch.log(relative_position_abs[is_large].float() / max_exact) / np.log(max_distance / max_exact) * (num_buckets_half - max_exact)).long()
    rp_bucket[is_large] += val_large.clamp(max=num_buckets-1)
    
    return rp_bucket

# 10 token'lık bir dizi için simülasyon yapalım
seq_len = 10
print(f"\n{seq_len} token'lık bir dizi için simülasyon yapılıyor...")

bucket_indices = compute_t5_relative_bias(seq_len)
print(f"\nAdım 4.1 & 4.2: Tokenlar arası mesafelerin 'kova' indeksleri:\n{bucket_indices}")

# Adım 4.3: Her kova için öğrenilmiş bias değerini tablodan al
bias = relative_attention_bias_table(bucket_indices).squeeze(-1)
print(f"\nAdım 4.3: Her pozisyon çifti için öğrenilmiş bias değerleri (rastgele):\n{bias.data.round(decimals=2)}")

# Adım 4.4: Bu bias'ı simüle edilmiş attention skorlarına ekle
dummy_attention_scores = torch.zeros(seq_len, seq_len)
final_scores_with_bias = dummy_attention_scores + bias

print("\nSONUÇ: Simüle edilmiş Attention Skorları + Göreceli Konum Bias'ı")
print(final_scores_with_bias.data.round(decimals=2))
print("\nSON YORUM: Dikkat ederseniz, matrisin diagonalleri boyunca aynı bias değerleri var.")
print("Örneğin, tüm ana diagonal (mesafe=0) aynı değere, bir üst diagonal (mesafe=-1) başka bir aynı değere sahip.")
print("Bu, yöntemin 'göreli' doğasını kanıtlar: Önemli olan mutlak pozisyon değil, aradaki mesafedir.")
print("-" * 60)

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

def get_sinusoidal_pe(max_len, d_model):
    """
    Sinusoidal Positional Encoding matrisini oluşturan fonksiyon.
    
    Args:
        max_len (int): Maksimum dizi uzunluğu (görselleştirilecek pozisyon sayısı).
        d_model (int): Modelin embedding boyutu.

    Returns:
        torch.Tensor: (max_len, d_model) boyutunda PE matrisi.
    """
    # Boş bir PE matrisi oluştur
    pe = torch.zeros(max_len, d_model)
    
    # Pozisyonları temsil eden bir vektör oluştur (0, 1, ..., max_len-1)
    position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
    
    # Dalga frekansını belirleyen terimi hesapla
    # Bu terim, boyut indeksi arttıkça daha uzun dalga boyları (düşük frekans) üretir
    div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-np.log(10000.0) / d_model))
    
    # Çift indeksli boyutlara sinüs fonksiyonunu uygula
    pe[:, 0::2] = torch.sin(position * div_term)
    
    # Tek indeksli boyutlara kosinüs fonksiyonunu uygula
    pe[:, 1::2] = torch.cos(position * div_term)
    
    return pe

# --- Görselleştirme Parametreleri ---
MAX_SEQ_LEN = 100  # Görselleştirilecek maksimum pozisyon sayısı
D_MODEL = 128      # Modelin embedding boyutu

# 1. PE matrisini oluştur
print("Sinusoidal Positional Encoding matrisi oluşturuluyor...")
pe_matrix = get_sinusoidal_pe(MAX_SEQ_LEN, D_MODEL)
print("Matris oluşturuldu.")

# 2. Matrisi bir ısı haritası (heatmap) olarak görselleştir
print("Görselleştirme oluşturuluyor...")
plt.figure(figsize=(12, 8))
sns.heatmap(pe_matrix.numpy(), cmap="viridis")

# Grafik başlığını ve eksen isimlerini ayarla
plt.title("Sinusoidal Positional Encoding Matrisi", fontsize=16)
plt.xlabel("Embedding Boyutları (d_model)", fontsize=12)
plt.ylabel("Pozisyon", fontsize=12)
plt.tight_layout()

# Görseli bir dosyaya kaydet
output_filename = "sinusoidal_pe_heatmap.png"
plt.savefig(output_filename)

print(f"\nBaşarılı! Görsel '{output_filename}' adıyla kaydedildi.")
print("Grafikteki her satır, bir pozisyon için benzersiz bir desene sahiptir.")